# Step 11: The Kitaev Chain — Topology and Majorana Edge Modes

**Quantum Simulation Series — Project 01: Quantum Criticality**

---

This is the final notebook in the series. It brings together everything that came before — exact diagonalisation, energy gaps, phase transitions, and tensor networks — while introducing something genuinely new: **topological order**.

The two phases of the Kitaev chain look identical from the perspective of any local order parameter. They have the same symmetry. The distinction between them is **global** — a property of the entire wavefunction that cannot be removed without closing the bulk energy gap. At the boundary between the two phases, **Majorana zero modes** appear: exotic quasiparticles that are their own antiparticles, non-locally encoded at the chain ends, and protected from local perturbations.

### What you will do
1. Build and diagonalise the Kitaev chain Bogoliubov–de Gennes (BdG) Hamiltonian
2. Map the phase diagram and identify the topological phase transition (gap closing)
3. Visualise Majorana edge mode wavefunctions
4. Compute the topological invariant (winding number) from the momentum-space Hamiltonian
5. Quantify ground state degeneracy splitting as a function of system size
6. Summarise the full series journey

### Prerequisites
Notebooks 05–10 (quantum Hilbert space, ED, entanglement, MPS, DMRG)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import eigh, eigvalsh
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print("NumPy", np.__version__)
try:
    import tenpy
    print("TeNPy", tenpy.__version__, "— available for DMRG cross-check")
    HAS_TENPY = True
except ImportError:
    print("TeNPy not installed — DMRG cells will be skipped")
    HAS_TENPY = False

## 11.1 The Kitaev Chain Hamiltonian

The Kitaev chain is a 1D lattice model of **spinless fermions** with three terms:

$$\hat{H} = -\mu \sum_{i=1}^{N} \hat{c}^\dagger_i \hat{c}_i - t \sum_{i=1}^{N-1} \left(\hat{c}^\dagger_{i+1} \hat{c}_i + \hat{c}_i^\dagger \hat{c}_{i+1}\right) + \Delta \sum_{i=1}^{N-1} \left(\hat{c}_{i+1} \hat{c}_i + \hat{c}^\dagger_i \hat{c}^\dagger_{i+1}\right)$$

- $\mu$: chemical potential — controls how many fermions are present
- $t > 0$: hopping amplitude — fermions move between sites
- $\Delta > 0$: p-wave pairing amplitude — pairs of fermions are created/annihilated simultaneously

The pairing term does not conserve particle number. This is the minimal model for a **1D topological superconductor**.

### Bogoliubov–de Gennes approach

Because the Hamiltonian is quadratic in fermion operators, it can be solved exactly for any $N$. Writing the Nambu spinor $\Psi = (c_1, \ldots, c_N, c^\dagger_1, \ldots, c^\dagger_N)^\intercal$, the Hamiltonian takes the form

$$\hat{H} = \frac{1}{2}\, \Psi^\dagger\, H_{\text{BdG}}\, \Psi + \text{const}$$

where the $2N \times 2N$ BdG Hamiltonian is

$$H_{\text{BdG}} = \begin{pmatrix} A & B \\ -B & -A \end{pmatrix}$$

with $A_{ij} = -\mu\, \delta_{ij} - t\,(\delta_{i,j+1} + \delta_{i,j-1})$ and $B_{ij} = \Delta\,(\delta_{i,j-1} - \delta_{i,j+1})$.

The eigenvalues of $H_{\text{BdG}}$ come in $\pm\varepsilon_\alpha$ pairs. The ground state energy is $E_0 = -\frac{1}{2}\sum_{\varepsilon_\alpha > 0} \varepsilon_\alpha$ (all negative-energy quasiparticle levels are filled).

In [ ]:
def build_bdg(N, mu, t=1.0, Delta=1.0):
    """
    Build the 2N×2N Bogoliubov-de Gennes Hamiltonian for the Kitaev chain (OBC).

    Nambu basis: Psi = (c_1,...,c_N, c†_1,...,c†_N)^T
    H_BdG = [[A, B], [-B, -A]]

    A_ij = -mu*delta_ij - t*(delta_{i,j+1} + delta_{i,j-1})   (hopping + chemical potential)
    B_ij = Delta*(delta_{i,j-1} - delta_{i,j+1})               (p-wave pairing, antisymmetric)
    """
    A = np.zeros((N, N))
    B = np.zeros((N, N))

    for i in range(N):
        A[i, i] = -mu
        if i + 1 < N:
            A[i, i+1] = -t
            A[i+1, i] = -t
            B[i, i+1] = -Delta   # B_{i,i+1} = -Delta  (delta_{i,j-1} with j=i+1 → i=j-1 ✓)
            B[i+1, i] = +Delta   # antisymmetric

    H = np.zeros((2*N, 2*N))
    H[:N, :N] = A
    H[:N, N:] = B
    H[N:, :N] = -B
    H[N:, N:] = -A
    return H


def solve_bdg(N, mu, t=1.0, Delta=1.0):
    """
    Return sorted positive quasi-particle energies and full eigensystem.
    Eigenvalues come in ±ε pairs; we return the full sorted array.
    """
    H = build_bdg(N, mu, t, Delta)
    evals, evecs = eigh(H)   # eigh for real symmetric (BdG is real symmetric here)
    return evals, evecs, H


def gap(N, mu, t=1.0, Delta=1.0):
    """Bulk gap = smallest positive BdG eigenvalue."""
    evals, _, _ = solve_bdg(N, mu, t, Delta)
    pos = evals[evals > 1e-12]
    return float(pos.min()) if len(pos) > 0 else 0.0


def ground_state_energy(N, mu, t=1.0, Delta=1.0):
    """Ground state energy E0 = -sum of positive quasi-particle energies."""
    evals, _, _ = solve_bdg(N, mu, t, Delta)
    return float(-np.sum(evals[evals > 0]))


# ── Quick sanity check on N=4 ──────────────────────────────────────────────
N = 4
H4 = build_bdg(N, mu=0.0, t=1.0, Delta=1.0)
print(f"H_BdG shape for N={N}: {H4.shape}")
print("Is real symmetric:", np.allclose(H4, H4.T))
print("Particle-hole symmetry (anti-unitary): H_BdG + τ_x H_BdG^T τ_x = 0?",
      np.allclose(H4 + np.block([[np.zeros((N,N)), np.eye(N)],
                                  [np.eye(N), np.zeros((N,N))]]) @
                  H4.T @
                  np.block([[np.zeros((N,N)), np.eye(N)],
                             [np.eye(N), np.zeros((N,N))]]), 0))

evals4, _, _ = solve_bdg(N, mu=0.0, t=1.0, Delta=1.0)
print(f"\nBdG eigenvalues (μ=0, t=Δ=1, N={N}): {np.round(evals4, 4)}")
print("(Should come in ±ε pairs; near-zero pair = Majorana modes)")

## 11.2 The Two Phases: Spectrum and Gap Closing

The Kitaev chain has two distinct gapped phases, separated by a **topological phase transition** at $|\mu| = 2t$.

**Trivial phase** ($|\mu| > 2t$):
- Unique gapped ground state
- No zero-energy modes
- Ground state is adiabatically connected to the vacuum (empty chain)

**Topological phase** ($|\mu| < 2t$, $\Delta \neq 0$):
- Ground state is **two-fold degenerate** (in the thermodynamic limit)
- Two near-zero-energy BdG eigenvalues — the Majorana zero modes
- These modes are localised at the chain ends (bulk-boundary correspondence)

The transition at $|\mu| = 2t$ is marked by the **bulk gap closing to zero**. This gap closing is mandatory: you cannot go from winding number $W = 0$ to $W = 1$ without the Hamiltonian passing through a gapless point.

Unlike the Ising/TFIM transition, there is **no local order parameter** distinguishing the two phases.

In [ ]:
N = 40
t, Delta = 1.0, 1.0
mu_values = np.linspace(-4, 4, 300)

# Collect the 4 lowest positive eigenvalues as a function of mu
spectra = []
for mu in mu_values:
    evals, _, _ = solve_bdg(N, mu, t, Delta)
    pos = np.sort(evals[evals > -1e-9])[:4]
    spectra.append(pos)
spectra = np.array(spectra)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: low-energy spectrum
ax = axes[0]
for k in range(4):
    ax.plot(mu_values / t, spectra[:, k],
            lw=1.5 if k > 1 else 2.5,
            color='steelblue' if k > 1 else ('crimson' if k == 0 else 'darkorange'),
            label=f'ε_{k+1}' if k < 3 else None)
ax.axvline(-2, color='gray', ls='--', lw=1)
ax.axvline(+2, color='gray', ls='--', lw=1)
ax.axhline(0, color='k', lw=0.5)
ax.fill_betweenx([0, 2.5], -2, 2, alpha=0.08, color='green', label='Topological')
ax.fill_betweenx([0, 2.5], -4, -2, alpha=0.08, color='orange', label='Trivial')
ax.fill_betweenx([0, 2.5], 2, 4, alpha=0.08, color='orange')
ax.set_xlabel(r'$\mu / t$')
ax.set_ylabel(r'Quasi-particle energy $\varepsilon_\alpha$')
ax.set_title(f'BdG spectrum vs $\\mu$ ($N={N}$, $\\Delta/t={Delta}$)')
ax.set_xlim(-4, 4)
ax.set_ylim(0, 2.5)
ax.legend(loc='upper right')
ax.text(0, 0.1, 'near-zero\nMajorana modes', ha='center', fontsize=9, color='crimson')

# Right: bulk gap Δ_gap vs μ/t — showing gap closing at |μ| = 2t
gaps = np.array([gap(N, mu, t, Delta) for mu in mu_values])
ax = axes[1]
ax.plot(mu_values / t, gaps, 'k-', lw=2, label='Bulk gap $\\Delta_{\\rm gap}$')
ax.axvline(-2, color='gray', ls='--', lw=1, label='$|\\mu|=2t$ (transition)')
ax.axvline(+2, color='gray', ls='--', lw=1)
ax.set_xlabel(r'$\mu / t$')
ax.set_ylabel(r'$\Delta_{\rm gap}$')
ax.set_title('Bulk gap — closes at the topological transition')
ax.legend()
ax.set_xlim(-4, 4)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

# Print gap at representative points
for mu_check in [-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0]:
    g = gap(N, mu_check, t, Delta)
    phase = 'trivial' if abs(mu_check) > 2*t else 'topological'
    print(f"  μ/t = {mu_check:+.1f}  →  gap = {g:.4f}  ({phase})")

## 11.3 Majorana Edge Modes

### The Majorana decomposition

Every complex fermion $\hat{c}_i$ can be written in terms of two Majorana operators:
$$\hat{c}_i = \tfrac{1}{2}(\hat{\gamma}_{2i-1} + i\,\hat{\gamma}_{2i}), \qquad
\hat{c}^\dagger_i = \tfrac{1}{2}(\hat{\gamma}_{2i-1} - i\,\hat{\gamma}_{2i})$$

Each Majorana satisfies $\hat{\gamma}_\alpha = \hat{\gamma}^\dagger_\alpha$ (it is its own antiparticle) and $\{\hat{\gamma}_\alpha, \hat{\gamma}_\beta\} = 2\delta_{\alpha\beta}$.

### What happens in each phase

**Trivial phase ($|\mu| \gg 2t$):** The dominant term in the Hamiltonian is the chemical potential $-\mu \hat{c}^\dagger_i \hat{c}_i$, which in Majorana language becomes $\sim i\mu\, \hat{\gamma}_{2i-1}\hat{\gamma}_{2i}$. Each site's two Majoranas pair *with each other* — the complex fermion on each site is intact.

**Topological sweet spot ($\mu = 0$, $|\Delta| = t$):** The Hamiltonian becomes exactly
$$\hat{H} = -2t \sum_{i=1}^{N-1} i\, \hat{\gamma}_{2i}\, \hat{\gamma}_{2i+1}$$
The pairings are now *between* nearest-neighbour sites: $\hat{\gamma}_{2i}$ on site $i$ pairs with $\hat{\gamma}_{2i+1}$ on site $i+1$. The two end Majoranas $\hat{\gamma}_1$ (left end) and $\hat{\gamma}_{2N}$ (right end) **do not appear in the Hamiltonian at all** — they are exact zero modes.

### Reading off the edge mode from the BdG eigenvectors

The zero-mode BdG eigenvector $v = (u_1,\ldots,u_N,\; v_1,\ldots,v_N)^\intercal$ gives the particle and hole components at each site. The **Majorana amplitude** at site $i$ is:
$$|\psi_{\rm MZM}(i)|^2 = u_i^2 + v_i^2$$

In the topological phase this is peaked at the chain ends; in the trivial phase it is spread through the bulk.

In [ ]:
def majorana_amplitude(N, mu, t=1.0, Delta=1.0):
    """
    Return the Majorana zero-mode amplitude |ψ(i)|² at each site.
    Uses the near-zero BdG eigenstate (smallest |ε|).
    """
    evals, evecs, _ = solve_bdg(N, mu, t, Delta)
    # Pick the eigenstate with smallest absolute eigenvalue
    idx = np.argmin(np.abs(evals))
    v = evecs[:, idx]
    u_part = v[:N]   # particle component
    v_part = v[N:]   # hole component
    amp = u_part**2 + v_part**2
    return amp / amp.sum()   # normalise


N = 60
sites = np.arange(1, N+1)

cases = [
    (0.0,  r'Topological sweet spot: $\mu=0$, $|\Delta|=t$', 'green'),
    (1.5,  r'Topological phase: $\mu/t=1.5$', 'steelblue'),
    (2.0,  r'Critical point: $\mu/t=2.0$ (gap=0)', 'purple'),
    (3.0,  r'Trivial phase: $\mu/t=3.0$', 'darkorange'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()

for ax, (mu_val, label, color) in zip(axes, cases):
    amp = majorana_amplitude(N, mu_val, t=1.0, Delta=1.0)
    ax.bar(sites, amp, color=color, alpha=0.7, width=1.0)
    ax.set_xlabel('Site $i$')
    ax.set_ylabel(r'$|\psi_{\rm MZM}(i)|^2$')
    ax.set_title(label, fontsize=10)
    ax.set_xlim(0.5, N + 0.5)

    # Annotate end weight
    end_weight = float(amp[0] + amp[-1] + amp[1] + amp[-2])
    ax.text(0.97, 0.95, f'Edge weight: {end_weight:.1%}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9)

plt.suptitle(f'Majorana zero-mode wavefunction ($N={N}$, $t=\\Delta=1$)', fontsize=12)
plt.tight_layout()
plt.show()

# Exponential decay length in topological phase
amp_topo = majorana_amplitude(N, mu=0.0, t=1.0, Delta=1.0)
half = N // 2
log_amp = np.log(amp_topo[:half] + 1e-20)
fit = np.polyfit(np.arange(half), log_amp, 1)
xi = -1.0 / fit[0]
print(f"\nExponential decay length in topological phase (μ=0): ξ ≈ {xi:.2f} sites")
print("(Majorana wavefunction decays as exp(-i/ξ) from the left end)")

## 11.4 Topological Invariant — The Winding Number

### Why a topological invariant?

The trivial and topological phases cannot be distinguished by any local order parameter. Instead, they are characterised by a **topological invariant** — a quantity computed from the entire momentum-space Hamiltonian that can only change when the gap closes.

For the Kitaev chain (class D in the Altland–Zirnbauer classification), the invariant is a $\mathbb{Z}_2$ number (0 or 1). It can be computed as a **winding number** of the 2D d-vector traced out by the momentum-space BdG Hamiltonian.

### Momentum-space BdG Hamiltonian (PBC)

For a periodic chain of length $N$, Fourier-transforming gives a $2\times2$ BdG Hamiltonian at each momentum $k = 2\pi n / N$:

$$H(k) = (-2t\cos k - \mu)\,\tau_z + 2\Delta\sin k\,\tau_y$$

$$= \begin{pmatrix} -2t\cos k - \mu & -2i\Delta\sin k \\ 2i\Delta\sin k & 2t\cos k + \mu \end{pmatrix}$$

This defines a 2D **d-vector** $\mathbf{d}(k) = (d_z, d_y) = (-2t\cos k - \mu,\; 2\Delta\sin k)$.

As $k$ traverses the full Brillouin zone $[0, 2\pi)$, the d-vector traces a closed curve in 2D.

### Winding number

$$W = \frac{1}{2\pi} \oint_{\rm BZ} dk\; \frac{d_z\, \partial_k d_y - d_y\, \partial_k d_z}{d_z^2 + d_y^2}$$

- **Trivial phase** ($|\mu| > 2t$): the curve does not encircle the origin → $W = 0$
- **Topological phase** ($|\mu| < 2t$): the curve encircles the origin once → $W = 1$
- **At the transition** ($|\mu| = 2t$): the curve passes through the origin — the gap closes and $W$ is undefined

The winding number is a topological invariant: it cannot change under any continuous deformation of the Hamiltonian that keeps the gap open.

In [ ]:
def winding_number(mu, t=1.0, Delta=1.0, nk=2000):
    """
    Compute the winding number of the d-vector for the Kitaev chain (PBC).

    d(k) = (-2t cos k - mu,  2*Delta sin k)   [z, y components]

    W = (1/2π) ∮ dk [dz * ∂k(dy) - dy * ∂k(dz)] / (dz² + dy²)
      = (1/2π) * total change in angle θ(k) = arctan(dy/dz)
    """
    k = np.linspace(0, 2*np.pi, nk, endpoint=False)
    dz = -2*t*np.cos(k) - mu
    dy = 2*Delta*np.sin(k)

    # Winding = total angle accumulated / 2π
    theta = np.arctan2(dy, dz)
    # unwrap to get continuous angle
    theta_unwrap = np.unwrap(theta)
    W = (theta_unwrap[-1] - theta_unwrap[0]) / (2*np.pi)
    return float(round(W))   # round to nearest integer


# Visualise d-vector trajectories and winding number
mu_cases = [0.0, 1.0, 1.99, 2.0, 2.01, 3.0]
t, Delta = 1.0, 1.0
nk = 500
k_arr = np.linspace(0, 2*np.pi, nk)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()

for ax, mu_val in zip(axes, mu_cases):
    dz = -2*t*np.cos(k_arr) - mu_val
    dy = 2*Delta*np.sin(k_arr)
    W = winding_number(mu_val, t, Delta)
    phase_label = 'trivial' if abs(mu_val) > 2*t else ('topological' if abs(mu_val) < 2*t else 'critical')
    color = {'topological': 'green', 'trivial': 'darkorange', 'critical': 'purple'}[phase_label]

    ax.plot(dz, dy, color=color, lw=2)
    ax.plot(0, 0, 'k+', ms=10, mew=2)  # mark origin
    ax.set_aspect('equal')
    ax.set_xlabel(r'$d_z(k)$')
    ax.set_ylabel(r'$d_y(k)$')
    ax.set_title(f'$\\mu/t={mu_val}$ — W={W}\n({phase_label})', fontsize=9)
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.text(0.03, 0.96, f'W = {int(W)}', transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top', color=color)

plt.suptitle('d-vector trajectory in the Brillouin zone ($t=\\Delta=1$)', fontsize=11)
plt.tight_layout()
plt.show()

# Winding number scan
print("\nWinding number vs μ/t:")
print(f"{'μ/t':>8}  {'W':>4}  {'gap':>8}  phase")
print("-" * 40)
for mu_val in [-3.5, -2.5, -2.0, -1.0, 0.0, 1.0, 2.0, 2.5, 3.5]:
    W = winding_number(mu_val, t, Delta)
    g = gap(40, mu_val, t, Delta)
    phase = 'trivial' if abs(mu_val) > 2*t else ('topological' if abs(mu_val) < 2*t else 'critical')
    print(f"{mu_val:>8.1f}  {int(W):>4}  {g:>8.4f}  {phase}")

## 11.5 Full Phase Diagram in the $(\mu/t,\, \Delta/t)$ Plane

The topological phase is bounded by the lines $|\mu| = 2t$ (for any $\Delta \neq 0$). The complete phase diagram in the $(\mu/t, \Delta/t)$ plane has:

- **Topological region** ($|\mu/t| < 2$, $\Delta \neq 0$): $W = 1$, two Majorana zero modes
- **Trivial region** ($|\mu/t| > 2$): $W = 0$, no edge modes
- **Phase boundary**: $|\mu| = 2t$ — gap closes along these vertical lines
- **$\Delta = 0$ line**: no pairing, no superconductivity, no topology (even for $|\mu| < 2t$)

We visualise this by plotting:
1. The bulk gap $\Delta_{\text{gap}}(\mu, \Delta)$ as a colour map
2. The winding number as a contour overlay
3. The ground-state degeneracy splitting as a proxy for edge mode presence

In [ ]:
N_pd = 40
n_mu, n_Delta = 80, 60
mu_arr = np.linspace(-4, 4, n_mu)
Delta_arr = np.linspace(0.0, 2.5, n_Delta)

gap_grid = np.zeros((n_Delta, n_mu))
wind_grid = np.zeros((n_Delta, n_mu))

for j, Delta_val in enumerate(Delta_arr):
    for i, mu_val in enumerate(mu_arr):
        gap_grid[j, i] = gap(N_pd, mu_val, t=1.0, Delta=Delta_val)
        wind_grid[j, i] = winding_number(mu_val, t=1.0, Delta=Delta_val)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: bulk gap colour map ---
ax = axes[0]
im = ax.pcolormesh(mu_arr, Delta_arr, gap_grid, cmap='viridis', vmin=0, vmax=2)
plt.colorbar(im, ax=ax, label='Bulk gap $\\Delta_{\\rm gap}$')
ax.axvline(-2, color='white', ls='--', lw=1.5, label='$|\\mu|=2t$')
ax.axvline(+2, color='white', ls='--', lw=1.5)
ax.set_xlabel(r'$\mu / t$')
ax.set_ylabel(r'$\Delta / t$')
ax.set_title('Bulk gap $\\Delta_{\\rm gap}(\\mu, \\Delta)$')
ax.legend(loc='upper right')
ax.text(0.0, 1.5, 'Topological\n$W=1$', ha='center', va='center', color='white', fontsize=11, fontweight='bold')
ax.text(-3.0, 1.5, 'Trivial\n$W=0$', ha='center', va='center', color='white', fontsize=11, fontweight='bold')
ax.text(+3.0, 1.5, 'Trivial\n$W=0$', ha='center', va='center', color='white', fontsize=11, fontweight='bold')

# --- Right: winding number ---
ax = axes[1]
# Replace non-integer values near the transition boundary with NaN for clean display
W_plot = wind_grid.copy()
im2 = ax.pcolormesh(mu_arr, Delta_arr, W_plot,
                    cmap='RdYlGn', vmin=-0.1, vmax=1.1)
plt.colorbar(im2, ax=ax, label='Winding number $W$')
ax.axvline(-2, color='k', ls='--', lw=1.5)
ax.axvline(+2, color='k', ls='--', lw=1.5)
ax.set_xlabel(r'$\mu / t$')
ax.set_ylabel(r'$\Delta / t$')
ax.set_title('Topological invariant (winding number $W$)')
ax.text(0.0, 1.5, '$W=1$', ha='center', va='center', color='k', fontsize=13, fontweight='bold')
ax.text(-3.2, 1.5, '$W=0$', ha='center', va='center', color='k', fontsize=13, fontweight='bold')
ax.text(+3.2, 1.5, '$W=0$', ha='center', va='center', color='k', fontsize=13, fontweight='bold')

for ax in axes:
    ax.set_xlim(-4, 4)
    ax.set_ylim(0, 2.5)

plt.tight_layout()
plt.show()

print("Phase diagram key points:")
print(f"  Gap at (μ=0, Δ=1):   {gap(N_pd, 0.0, 1.0, 1.0):.4f}  (topological)")
print(f"  Gap at (μ=0, Δ=0):   {gap(N_pd, 0.0, 1.0, 0.0):.4f}  (no pairing → gapless band)")
print(f"  Gap at (μ=3, Δ=1):   {gap(N_pd, 3.0, 1.0, 1.0):.4f}  (trivial)")
print(f"  Gap at (μ=2, Δ=1):   {gap(N_pd, 2.0, 1.0, 1.0):.4f}  (transition)")

## 11.6 Ground State Degeneracy and the Non-Local Qubit

### The two-fold degeneracy

In the topological phase, the two Majorana zero modes $\hat{\gamma}_1$ and $\hat{\gamma}_{2N}$ can be combined into one complex fermion:

$$\hat{f} = \frac{1}{2}(\hat{\gamma}_1 + i\, \hat{\gamma}_{2N}), \qquad \hat{f}^\dagger \hat{f} = 0 \text{ or } 1$$

This fermion does not appear in the bulk Hamiltonian. Its occupation number is not fixed — both $|f=0\rangle$ and $|f=1\rangle$ are ground states. The ground state is **exactly two-fold degenerate** in the thermodynamic limit.

For a finite chain, the two Majorana wavefunctions have exponentially small overlap $\sim e^{-N/\xi}$, which gives an exponentially small energy splitting:

$$\delta E = E_1 - E_0 \sim e^{-N/\xi}$$

This is the physical protection: any local error would need to tunnel from one end to the other, costing energy $e^{-N/\xi}$.

### The non-local qubit

The two ground states form a **qubit**: $|0\rangle$ (fermion empty) and $|1\rangle$ (fermion occupied). The key property: the two basis states differ only in the occupation of a mode delocalised over the *entire* chain. Any local operation on one end of the chain is equivalent to acting on only one of the two Majorana operators — it cannot distinguish or rotate the qubit state.

This **non-locality** is the basis for topological quantum computation: quantum information encoded in pairs of Majorana zero modes is automatically protected from local decoherence.

In [ ]:
def degeneracy_splitting(N, mu, t=1.0, Delta=1.0):
    """
    Return δE = |ε_0 - ε_1|, the splitting of the two near-zero BdG eigenvalues.
    In the topological phase these are the Majorana zero-mode energies.
    """
    evals, _, _ = solve_bdg(N, mu, t, Delta)
    sorted_abs = np.sort(np.abs(evals))
    return float(sorted_abs[0] + sorted_abs[1])   # sum of two smallest |ε|


t, Delta = 1.0, 1.0
chain_sizes = np.arange(10, 101, 5)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# --- Panel 1: δE vs N for different μ in topological phase ---
ax = axes[0]
for mu_val, color, ls in [(0.0, 'green', '-'), (1.0, 'steelblue', '--'), (1.8, 'purple', ':')]:
    splittings = [degeneracy_splitting(N, mu_val, t, Delta) for N in chain_sizes]
    ax.semilogy(chain_sizes, splittings, color=color, ls=ls, lw=2,
                label=f'$\\mu/t={mu_val}$')
ax.set_xlabel('Chain length $N$')
ax.set_ylabel(r'$\delta E = |\varepsilon_0| + |\varepsilon_1|$')
ax.set_title('Degeneracy splitting vs $N$\n(topological phase)')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Panel 2: Extract correlation length ξ from exponential fit ---
ax = axes[1]
mu_scan = np.linspace(0.1, 1.9, 40)
N_fit = 80
xi_vals = []
for mu_val in mu_scan:
    Ns = np.arange(20, N_fit+1, 5)
    splits = np.array([degeneracy_splitting(n, mu_val, t, Delta) for n in Ns])
    mask = splits > 1e-14
    if mask.sum() > 3:
        log_s = np.log(splits[mask])
        slope, _ = np.polyfit(Ns[mask], log_s, 1)
        xi_vals.append(-1.0 / slope)
    else:
        xi_vals.append(np.nan)
xi_vals = np.array(xi_vals)

ax.plot(mu_scan, xi_vals, 'k-o', ms=3, lw=1.5)
ax.set_xlabel(r'$\mu / t$')
ax.set_ylabel(r'Decay length $\xi$')
ax.set_title('Majorana decay length $\\xi$ vs $\\mu/t$\n(diverges at transition $\\mu/t=2$)')
ax.axvline(2.0, color='red', ls='--', lw=1, label='$\\mu/t=2$')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0)

# --- Panel 3: Comparison trivial vs topological ---
ax = axes[2]
for mu_val, label, color in [(0.0, 'topo $\\mu/t=0$', 'green'),
                               (3.0, 'trivial $\\mu/t=3$', 'darkorange'),
                               (2.0, 'critical $\\mu/t=2$', 'purple')]:
    splittings = [degeneracy_splitting(N, mu_val, t, Delta) for N in chain_sizes]
    ax.semilogy(chain_sizes, splittings, lw=2, color=color, label=label)
ax.set_xlabel('Chain length $N$')
ax.set_ylabel(r'$\delta E$')
ax.set_title('Trivial vs topological:\ndegeneracy splitting')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Exponential fit for μ=0 case
Ns_fit = np.arange(20, 101, 5)
splits_0 = np.array([degeneracy_splitting(n, 0.0, t, Delta) for n in Ns_fit])
slope, intercept = np.polyfit(Ns_fit, np.log(splits_0 + 1e-20), 1)
xi_fit = -1.0 / slope
print(f"Exponential fit for μ=0, Δ=t=1:")
print(f"  δE ~ exp(-N/ξ),  ξ ≈ {xi_fit:.2f} sites")
print(f"  (Analytical: ξ = 1/log(t/Δ) → ∞ at μ=0, t=Δ; ξ ≈ 1/|log(μ/2t)| for small μ)")

## 11.7 Bulk-Boundary Correspondence Verified

The **bulk-boundary correspondence** is the central theorem of topological phases: the topological invariant of the *bulk* Hamiltonian uniquely determines the number of protected *boundary* modes.

For the Kitaev chain:
$$W_{\rm bulk} = \begin{cases} 0 & |\mu| > 2t \quad \text{(trivial)} \\ 1 & |\mu| < 2t \quad \text{(topological)} \end{cases}
\quad \Longleftrightarrow \quad
\text{number of Majorana zero modes at each end} = W_{\rm bulk}$$

We verify this directly by comparing:
- The winding number computed from the PBC momentum-space Hamiltonian (bulk property)
- The number of near-zero BdG eigenvalues with OBC (boundary property)

The two must agree whenever the bulk gap is open — this is the content of the bulk-boundary correspondence.

In [ ]:
def count_zero_modes(N, mu, t=1.0, Delta=1.0, tol=0.05):
    """
    Count the number of near-zero BdG eigenvalues with OBC.
    'Near-zero' means |ε| < tol * bulk_gap.
    """
    evals, _, _ = solve_bdg(N, mu, t, Delta)
    g = gap(N, mu, t, Delta)
    if g < 1e-10:
        return -1  # gapless, undefined
    threshold = max(tol * g, 1e-6)
    return int(np.sum(np.abs(evals) < threshold))


N_bbc = 60
t, Delta = 1.0, 1.0
mu_test = np.linspace(-4, 4, 200)

winding = np.array([winding_number(m, t, Delta) for m in mu_test])
n_modes = np.array([count_zero_modes(N_bbc, m, t, Delta) for m in mu_test])
gap_arr  = np.array([gap(N_bbc, m, t, Delta) for m in mu_test])

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

ax = axes[0]
ax.plot(mu_test, winding, 'b-', lw=2, label='Winding number $W$ (bulk, PBC)')
ax.set_ylabel('$W$')
ax.set_ylim(-0.2, 1.5)
ax.set_yticks([0, 1])
ax.legend(loc='upper right')
ax.axvline(-2, color='gray', ls='--', lw=1)
ax.axvline(+2, color='gray', ls='--', lw=1)
ax.set_title(f'Bulk-Boundary Correspondence — Kitaev chain ($N={N_bbc}$, $t=\\Delta=1$)')

ax = axes[1]
mask_gapped = n_modes >= 0
ax.scatter(mu_test[mask_gapped], n_modes[mask_gapped] // 2, s=8, c='crimson',
           label='Majorana modes per end (OBC)')
ax.set_ylabel('MZM per end')
ax.set_ylim(-0.2, 1.5)
ax.set_yticks([0, 1])
ax.legend(loc='upper right')
ax.axvline(-2, color='gray', ls='--', lw=1)
ax.axvline(+2, color='gray', ls='--', lw=1)

ax = axes[2]
ax.plot(mu_test, gap_arr, 'k-', lw=2, label='Bulk gap $\\Delta_{\\rm gap}$')
ax.set_ylabel('Gap')
ax.set_xlabel(r'$\mu / t$')
ax.legend(loc='upper right')
ax.axvline(-2, color='gray', ls='--', lw=1, label='$|\\mu|=2t$')
ax.axvline(+2, color='gray', ls='--', lw=1)
ax.set_ylim(bottom=0)

for ax in axes:
    ax.fill_betweenx(ax.get_ylim(), -2, 2, alpha=0.05, color='green')

plt.tight_layout()
plt.show()

# Verification table
print(f"\nBulk-boundary correspondence check (N={N_bbc}):")
print(f"{'μ/t':>6}  {'W (bulk)':>10}  {'MZM/end (OBC)':>14}  {'gap':>8}  {'agree?':>7}")
print("-" * 55)
for mu_val in [-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0]:
    W = int(winding_number(mu_val, t, Delta))
    n = count_zero_modes(N_bbc, mu_val, t, Delta)
    mzm_per_end = n // 2 if n >= 0 else '?'
    g = gap(N_bbc, mu_val, t, Delta)
    agree = '✓' if (n < 0 or int(mzm_per_end) == W) else '✗'
    print(f"{mu_val:>6.1f}  {W:>10}  {str(mzm_per_end):>14}  {g:>8.4f}  {agree:>7}")

## 11.8 Series Summary — Where You Have Arrived

The Kitaev chain brings the series to a natural close. Here is the full journey:

| Step | Model | Physics | Method |
|------|-------|---------|--------|
| 01 | 2D Ising | Thermal phase transition, spontaneous symmetry breaking | Metropolis Monte Carlo |
| 02 | 2D Ising (FSS) | Critical exponents, universality classes | Finite-size scaling, Binder cumulant |
| 03 | XY model | Topological defects, BKT transition (no Landau order) | Monte Carlo + vortex counting |
| 04 | Ising + field | First-order transitions, Landau free energy | Monte Carlo |
| 05 | Heisenberg (quantum) | Many-body Hilbert space, quantum fluctuations | Exact diagonalisation |
| 06 | TFIM | Quantum phase transition, gap closing, Hamiltonian ED | ED with sparse matrices |
| 07 | TFIM (FSS) | Quantum critical exponents, spectral gap, scaling collapse | ED + FSS |
| 08 | TFIM/Heisenberg | Entanglement entropy, area law, CFT central charge | ED + Schmidt decomposition |
| 09 | TFIM | Matrix product states, SVD truncation, canonical form | MPS |
| 10 | TFIM/Heisenberg | DMRG, large-scale simulation, convergence | TeNPy |
| **11** | **Kitaev chain** | **Topological phase, Majorana zero modes, bulk-boundary** | **BdG + TeNPy** |

### The big arc: three types of phase transition

1. **Thermal transitions** (steps 01–04): Landau paradigm — order parameter, symmetry breaking, universality classes. The transition is driven by thermal fluctuations and described by a local order parameter.

2. **Quantum transitions** (steps 05–10): Zero-temperature phase transitions driven by quantum fluctuations. The gap closes at the transition; exponents obey quantum-to-classical mapping. Still within the Landau paradigm — phases differ by broken symmetry.

3. **Topological transitions** (step 11): Beyond Landau — phases cannot be distinguished by any local order parameter. The distinction is global, encoded in a topological invariant. Protected edge modes appear at boundaries. Relevant to quantum information and topological quantum computation.

### What comes next

The tools you have built — Monte Carlo, finite-size scaling, exact diagonalisation, tensor networks — are the same ones used in active research. Natural extensions include:

- **Higher-dimensional topological phases**: 2D Chern insulators, 3D topological insulators
- **Fractional quantum Hall effect**: topological order with anyonic excitations
- **Floquet topology**: topological phases driven by periodic driving
- **Topological quantum error correction**: surface codes and Majorana-based qubits
- **Interacting topological phases**: where tensor-network methods become essential

In [ ]:
# ── Final summary figure: all key results in one panel ───────────────────────
N_sum = 60
t, Delta_sum = 1.0, 1.0
mu_vals = np.linspace(-4, 4, 300)

gaps_sum  = np.array([gap(N_sum, m, t, Delta_sum) for m in mu_vals])
winds_sum = np.array([winding_number(m, t, Delta_sum) for m in mu_vals])
splits_sum = np.array([degeneracy_splitting(N_sum, m, t, Delta_sum) for m in mu_vals])

# Edge weight (fraction of MZM amplitude on the 4 outermost sites)
edge_weights = []
for m in mu_vals:
    amp = majorana_amplitude(N_sum, m, t, Delta_sum)
    edge_weights.append(amp[0] + amp[1] + amp[-2] + amp[-1])
edge_weights = np.array(edge_weights)

fig = plt.figure(figsize=(13, 9))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

# Panel A: Gap
ax = fig.add_subplot(gs[0, 0])
ax.plot(mu_vals, gaps_sum, 'k-', lw=2)
ax.axvline(-2, color='gray', ls='--', lw=1)
ax.axvline(+2, color='gray', ls='--', lw=1)
ax.fill_betweenx([0, 2.5], -2, 2, alpha=0.1, color='green')
ax.set_xlabel(r'$\mu/t$'); ax.set_ylabel(r'$\Delta_{\rm gap}$')
ax.set_title('(A) Bulk gap closes at $|\\mu|=2t$')
ax.set_ylim(0); ax.set_xlim(-4, 4)
ax.text(0, 1.5, 'Topological', ha='center', color='green', fontsize=9)
ax.text(-3, 1.5, 'Trivial', ha='center', color='darkorange', fontsize=9)

# Panel B: Winding number
ax = fig.add_subplot(gs[0, 1])
ax.step(mu_vals, winds_sum, 'b-', lw=2.5, where='mid')
ax.axvline(-2, color='gray', ls='--', lw=1)
ax.axvline(+2, color='gray', ls='--', lw=1)
ax.fill_betweenx([-0.1, 1.4], -2, 2, alpha=0.1, color='green')
ax.set_xlabel(r'$\mu/t$'); ax.set_ylabel(r'$W$')
ax.set_title('(B) Winding number (topological invariant)')
ax.set_ylim(-0.15, 1.3); ax.set_yticks([0, 1]); ax.set_xlim(-4, 4)

# Panel C: Degeneracy splitting (log scale)
ax = fig.add_subplot(gs[1, 0])
ax.semilogy(mu_vals, splits_sum + 1e-16, 'crimson', lw=2)
ax.axvline(-2, color='gray', ls='--', lw=1)
ax.axvline(+2, color='gray', ls='--', lw=1)
ax.fill_between(mu_vals, 1e-16, 3,
                where=(np.abs(mu_vals) < 2), alpha=0.1, color='green')
ax.set_xlabel(r'$\mu/t$'); ax.set_ylabel(r'$\delta E$ (log scale)')
ax.set_title(f'(C) GS degeneracy splitting ($N={N_sum}$)')
ax.set_xlim(-4, 4); ax.set_ylim(1e-14, 3)

# Panel D: Edge weight of the MZM
ax = fig.add_subplot(gs[1, 1])
ax.plot(mu_vals, edge_weights, 'darkorange', lw=2)
ax.axvline(-2, color='gray', ls='--', lw=1)
ax.axvline(+2, color='gray', ls='--', lw=1)
ax.fill_betweenx([0, 1], -2, 2, alpha=0.1, color='green')
ax.set_xlabel(r'$\mu/t$')
ax.set_ylabel('Edge weight of MZM')
ax.set_title(f'(D) Majorana mode localisation at ends ($N={N_sum}$)')
ax.set_xlim(-4, 4); ax.set_ylim(0, 1.05)

plt.suptitle('Kitaev Chain: Summary of Topological Phase Transition ($t=\\Delta=1$)',
             fontsize=13, fontweight='bold')
plt.show()

print("=" * 60)
print("  Quantum Simulation Series — COMPLETE")
print("=" * 60)
print(f"  11 notebooks | Monte Carlo → DMRG → Topology")
print(f"  Classical phase transitions → Quantum → Topological")